# E3 — seed stability on CIFAR-100

Reuses the three E1 seed-1 results and runs six new evaluations for seeds 2 and 3. Missing seed-2/3 backbones are trained for 200 epochs with the E1 training recipe and checkpointed each epoch; rerun the orchestration cell in the same working directory to resume an interrupted training run. The class-order seed remains 1993.

Before running: push `feat/e3-seed-stability` to GitHub, attach the E1 results as a Kaggle Dataset, and set `E1_ROOT` below. Enable Internet and a GPU. Group weight decay is 0.0005; the paper reports 0.0001 for ResNet-18, so this seed study is not an exact paper-hyperparameter reproduction.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/duongthienphu/fecam-study.git'
REPO_BRANCH = 'feat/e3-seed-stability'
REPO_DIR = Path('/kaggle/working/fecam-study')
if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets', 'PyYAML', 'matplotlib', 'wandb'], check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
# Attach the downloaded E1 result ZIP as a Kaggle Dataset and update this folder if needed.
E1_ROOT = Path('/kaggle/input/e1-seed1-results/e1-seed1')
SEED1_SUMMARIES = [
    E1_ROOT / 'ncm/summary.json',
    E1_ROOT / 'fecam_shared/summary.json',
    E1_ROOT / 'fecam_classwise/summary.json',
]
for path in SEED1_SUMMARIES:
    if not path.is_file():
        raise FileNotFoundError(f'Missing completed E1 seed-1 summary: {path}')
if not __import__('torch').cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook Settings')

OUTPUT_ROOT = Path('/kaggle/working/e3')
CHECKPOINT_TEMPLATE = str(OUTPUT_ROOT / 'backbones/seed{seed}/base.pt')
# Optional: attach a previous Kaggle output Dataset to resume saved base_training_checkpoint.pt files.
RESUME_ROOT = Path('/kaggle/input/e3-resume')
for seed in (2, 3):
    source = RESUME_ROOT / 'backbones' / f'seed{seed}'
    destination = OUTPUT_ROOT / 'backbones' / f'seed{seed}'
    if source.is_dir() and not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, destination)
        print(f'Restored seed-{seed} checkpoint/training state for resume.')

In [ ]:
command = [
    sys.executable, 'scripts/run_e3.py',
    '--seed1-summaries', *[str(path) for path in SEED1_SUMMARIES],
    '--checkpoint-template', CHECKPOINT_TEMPLATE,
    '--output-dir', str(OUTPUT_ROOT),
    '--train-missing-checkpoints', '--device', 'cuda:0',
]
subprocess.run(command, check=True)

all_summaries = [str(path) for path in SEED1_SUMMARIES]
for seed in (2, 3):
    all_summaries.extend(str(OUTPUT_ROOT / 'evaluations' / f'seed{seed}' / method / 'summary.json')
                        for method in ('ncm', 'fecam_shared', 'fecam_classwise'))

# Optional W&B upload using a Kaggle Secret named WANDB_API_KEY.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    print('WANDB_API_KEY secret unavailable; skipping W&B upload.')
else:
    subprocess.run([sys.executable, 'scripts/upload_e3_to_wandb.py', *all_summaries, '--report', str(OUTPUT_ROOT / 'report/e3_seed_stability.csv'), '--plot', str(OUTPUT_ROOT / 'report/e3_seed_stability.png')], check=True)

shutil.make_archive('/kaggle/working/e3-results-and-checkpoints', 'zip', root_dir='/kaggle/working', base_dir='e3')
print('Download e3-results-and-checkpoints.zip from Kaggle Output. It contains evaluation results and resumable backbone checkpoints.')